Here is how it works:
- it sorts and index the url also save the list of random sample url indeces
- after each puase or interuption the "Cloned Repo" should be emptyied
- by a new new rerun it continues the review from the last reviewed url which is log is stored in .evn by START_NUMBER
- the sample repos are stored in "Cloned_Sample"
- this will save the sample repos as well as metrics, configs, builds and test lines
- Full Clone helps to extract full contributors and commit history
- saving metadata happend immediately so it will get lost by pause/start

In [ ]:
import pandas as pd
import os
import subprocess
import shutil
import glob
import random
from pathlib import Path
from dotenv import load_dotenv, set_key
import requests

# === CONFIGURATION ===
MAX_PROJECTS = 5
RANDOM_SEED = 42
NUM_SAMPLES_TO_KEEP = 150
ENV_FILE = 'All_tokens.env'

# === LOAD .env ===
load_dotenv(ENV_FILE)
GITHUB_TOKEN = os.getenv('GITHUB_TOKEN')
if not GITHUB_TOKEN:
    raise ValueError("❌ GitHub token not found in All_tokens.env")

START_NUMBER = int(os.getenv("START_NUMBER"))
print("Start_number:", START_NUMBER)
SAMPLE_LIST_RAW = os.getenv("SAMPLE_LIST", "").strip()

# === PATHS ===
csv_path = r"C:\GitHub\Android-Mobile-Apps\8.1-Project_GitHub_URLs.csv"
base_dir = Path(r"E:\Android Mobile Project\AndroidProjects")
clone_dir = base_dir / "Cloned repos"
cloned_sample_dir = base_dir / "Cloned_Sample"
yml_output_dir = base_dir / "Config Files"
commits_dir = base_dir / "Commits"
build_info_dir = base_dir / "BuildInfo"
metadata_path = base_dir / "8.2 - Metadata.csv"

# === CLEAN OLD DATA ===
for path in [clone_dir, yml_output_dir, commits_dir, build_info_dir]:
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)

cloned_sample_dir.mkdir(parents=True, exist_ok=True)

# === LOAD AND CLEAN CSV ===
df = pd.read_csv(csv_path)
df.columns = df.columns.str.strip().str.lower()
df = df[df['github_url'].notna()]
df['github_url'] = df['github_url'].astype(str).str.strip()
df = df[df['github_url'].str.startswith("https://")]
df = df.sort_values(by='github_url').reset_index(drop=True)
df[['github_url']].to_csv(base_dir / 'Sorted_URL_List.csv', index_label='Index')

# === HANDLE SAMPLE_LIST ===
if SAMPLE_LIST_RAW:
    sample_indices_to_keep = set(map(int, SAMPLE_LIST_RAW.split(',')))
    print(f"🔁 Loaded SAMPLE_LIST from .env with {len(sample_indices_to_keep)} indices.")
else:
    random.seed(RANDOM_SEED)
    sample_indices_to_keep = set(random.sample(range(len(df)), min(NUM_SAMPLES_TO_KEEP, len(df))))
    sample_string = ",".join(map(str, sorted(sample_indices_to_keep)))
    set_key(ENV_FILE, 'SAMPLE_LIST', sample_string)
    print(f"🎲 Generated and saved new SAMPLE_LIST with {len(sample_indices_to_keep)} indices.")

# === PROCESS EACH REPO ===
print("Start_number: ", START_NUMBER)
for i in range(START_NUMBER - 1, len(df)):
    url = df.loc[i, 'github_url']
    parts = url.split('/')
    if len(parts) < 5:
        continue
    username, project = parts[-2], parts[-1].replace('.git', '')
    repo_index = str(i).zfill(4)
    repo_name = f"{repo_index}.{username}.{project}"
    repo_path = clone_dir / repo_name

    print(f"\n🔍 [{i+1}/{len(df)}] Processing {repo_name}...")

    try:
        subprocess.run(['git', 'clone', url, str(repo_path)], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    except subprocess.TimeoutExpired:
        print(f"⏱️ Timeout while cloning {repo_name}, skipping...")
        continue
    print("✅ Clone complete")

    for root, _, files in os.walk(repo_path):
        for file in files:
            if file.endswith(('.yml', '.yaml')):
                full_path = Path(root) / file
                rel_path = full_path.relative_to(repo_path)
                safe_name = f"{repo_name}.{str(rel_path).replace(os.sep, '_')}"
                shutil.copy2(full_path, yml_output_dir / safe_name)
    print("📄 Config files extracted")

    build_info_dir.mkdir(parents=True, exist_ok=True)
    info_path = build_info_dir / f"{repo_name}_build_info.txt"
    with open(info_path, 'w', encoding='utf-8') as out_file:
        for gradle_file in glob.glob(str(repo_path / '**/*.gradle*'), recursive=True):
            try:
                with open(gradle_file, 'r', encoding='utf-8', errors='ignore') as f:
                    lines = f.readlines()
                    test_lines = [line for line in lines if 'test' in line.lower()]
                    if test_lines:
                        out_file.write(f"\n--- {gradle_file} ---\n")
                        out_file.writelines(test_lines)
            except Exception:
                continue
    print("🛠️ Build info extracted")

    with open(commits_dir / f"{repo_name}_commits.txt", 'w', encoding='utf-8') as f:
        subprocess.run(['git', 'log', '--pretty=format:%h | %an | %ad | %s'], cwd=repo_path, stdout=f, stderr=subprocess.DEVNULL)
    print("📜 Commits saved")

    try:
        api_url = f"https://api.github.com/repos/{username}/{project}/contributors"
        r = requests.get(api_url, headers={'Authorization': f'token {GITHUB_TOKEN}'}, timeout=30)
        if r.ok:
            contributors_data = r.json()
            contrib_path = commits_dir / f"{repo_name}_contributors.txt"
            with open(contrib_path, 'w', encoding='utf-8') as f:
                for contributor in contributors_data:
                    f.write(f"{contributor['contributions']:>4} | {contributor['login']}\n")
            print("👥 Contributors saved via GitHub API")
        else:
            print(f"⚠️ GitHub API error ({r.status_code}) for contributors of {repo_name}")
    except Exception as e:
        print(f"⚠️ Error retrieving contributors for {repo_name}: {e}")

    try:
        r = requests.get(f"https://api.github.com/repos/{username}/{project}", headers={'Authorization': f'token {GITHUB_TOKEN}'}, timeout=30)
        if r.ok:
            data = r.json()

            pr_count = 0
            pr_url = f"https://api.github.com/repos/{username}/{project}/pulls?state=all&per_page=1"
            pr_resp = requests.get(pr_url, headers={'Authorization': f'token {GITHUB_TOKEN}'})
            if 'Link' in pr_resp.headers:
                pr_count = int(pr_resp.headers['Link'].split('page=')[-1].split('>')[0])
            else:
                pr_count = len(pr_resp.json())

            commit_count = 0
            commit_url = f"https://api.github.com/repos/{username}/{project}/commits?per_page=1"
            commit_resp = requests.get(commit_url, headers={'Authorization': f'token {GITHUB_TOKEN}'})
            if 'Link' in commit_resp.headers:
                commit_count = int(commit_resp.headers['Link'].split('page=')[-1].split('>')[0])
            else:
                commit_count = len(commit_resp.json())

            contrib_url = f"https://api.github.com/repos/{username}/{project}/contributors"
            contrib_resp = requests.get(contrib_url, headers={'Authorization': f'token {GITHUB_TOKEN}'})
            contrib_count = len(contrib_resp.json()) if contrib_resp.ok else 0

            new_row_df = pd.DataFrame([{
                'repo_name': repo_name,
                'full_name': data.get('full_name'),
                'description': data.get('description'),
                'language': data.get('language'),
                'license': data.get('license', {}).get('name') if data.get('license') else None,
                'created_at': data.get('created_at'),
                'updated_at': data.get('updated_at'),
                'last_commit_date': data.get('pushed_at'),
                'stars': data.get('stargazers_count'),
                'forks': data.get('forks_count'),
                'watchers': data.get('watchers_count'),
                'open_issues': data.get('open_issues_count'),
                'contributors': contrib_count,
                'pull_requests': pr_count,
                'commits': commit_count,
                'size': data.get('size')
            }])

            if metadata_path.exists():
                new_row_df.to_csv(metadata_path, mode='a', header=False, index=False)
            else:
                new_row_df.to_csv(metadata_path, mode='w', header=True, index=False)

            print("📋 Metadata appended to file")
    except Exception as e:
        print(f"⚠️ Exception retrieving metadata for {repo_name}: {e}")

    try:
        if i in sample_indices_to_keep:
            dest_path = cloned_sample_dir / repo_path.name
            if dest_path.exists():
                shutil.rmtree(dest_path, ignore_errors=True)
            shutil.move(str(repo_path), str(dest_path))
            print(f"📦 Sample repo moved to: {dest_path}")
        else:
            shutil.rmtree(repo_path, ignore_errors=True)
            print(f"🛑 Non-sample repo deleted: {repo_path.name}")
    except Exception as e:
        print(f"❌ Error handling repo folder for {repo_name}: {e}")

    set_key(ENV_FILE, 'START_NUMBER', str(i + 2))

print("\n🌟 Finished processing selected projects.")


Start_number: 295
🔁 Loaded SAMPLE_LIST from .env with 150 indices.
Start_number:  295

🔍 [295/1796] Processing 0294.MengTo.AppStoreSketch...
✅ Clone complete
📄 Config files extracted
🛠️ Build info extracted
📜 Commits saved
👥 Contributors saved via GitHub API
📊 Full metrics retrieved
🗑️ Non-sample repo deleted: 0294.MengTo.AppStoreSketch

🔍 [296/1796] Processing 0295.MeoMunDep.Cell-Wallet...
✅ Clone complete
📄 Config files extracted
🛠️ Build info extracted
📜 Commits saved
👥 Contributors saved via GitHub API
📊 Full metrics retrieved
🗑️ Non-sample repo deleted: 0295.MeoMunDep.Cell-Wallet

🔍 [297/1796] Processing 0296.MexsonFernandes.ClusTMPay-SIH2019...
✅ Clone complete
📄 Config files extracted
🛠️ Build info extracted
📜 Commits saved
👥 Contributors saved via GitHub API
📊 Full metrics retrieved
🗑️ Non-sample repo deleted: 0296.MexsonFernandes.ClusTMPay-SIH2019

🔍 [298/1796] Processing 0297.MiSTer-devel.Main_MiSTer...
✅ Clone complete
📄 Config files extracted
🛠️ Build info extracted
📜 Commi